# 3-Step Noise Regime Grid Search Analysis

Explores the parameter space of the **noisy perceptual traces** hypothesis:
recognition memory depends on the fidelity of noisy traces with an
age-dependent noise schedule. No prior-driven drift.

| Parameter | Role |
|-----------|------|
| `sigma0` | Encoding noise (applied once at memory insertion) |
| `sigma1` | Diffusive noise for recent memories (age < t_step) |
| `sigma2` | Diffusive noise for older memories (age >= t_step) |

This notebook loads merged grid search results and visualises how
different combinations of (sigma0, sigma1, sigma2) affect d', AUROC,
and score distributions across ISI conditions.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

from utls.roc_utils import roc_from_arrays
from utls.analysis_helpers import auroc_to_dprime

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

## 1. Load merged grid search results

In [ ]:
SAVE_DIR = '../reports/figures/3step_grid_search'
RESULTS_PATH = os.path.join(SAVE_DIR, 'grid_search_results_3step.npz')

data = np.load(RESULTS_PATH)

sigma0_grid = data['sigma0_grid']
sigma1_grid = data['sigma1_grid']
sigma2_grid = data['sigma2_grid']
isi_values = data['isi_values']

# 3D d' arrays: [n_s0, n_s1, n_s2]
dprime = {int(isi): data[f'dprime_isi{isi}'] for isi in isi_values}

print(f'sigma0 grid: {sigma0_grid}')
print(f'sigma1 grid: {sigma1_grid}')
print(f'sigma2 grid: {sigma2_grid}')
print(f'ISI values:  {isi_values}')
print(f'd\' array shape: {dprime[int(isi_values[0])].shape}')
print(f'Total configs: {np.prod(dprime[int(isi_values[0])].shape)}')

## 2. Build flat DataFrame (one row per triple)

In [ ]:
rows = []
for i_s0, s0 in enumerate(sigma0_grid):
    for i_s1, s1 in enumerate(sigma1_grid):
        for i_s2, s2 in enumerate(sigma2_grid):
            row = {
                'sigma0': s0, 'sigma1': s1, 'sigma2': s2,
                'i_s0': i_s0, 'i_s1': i_s1, 'i_s2': i_s2,
            }
            dp_vals = []
            for isi in isi_values:
                dp = dprime[int(isi)][i_s0, i_s1, i_s2]
                row[f'dprime_isi{isi}'] = dp
                dp_vals.append(dp)
            row['dprime_mean'] = np.nanmean(dp_vals)
            # ISI decay: d'(ISI_min) - d'(ISI_max)
            row['delta_dp'] = dp_vals[0] - dp_vals[-1] if len(dp_vals) > 1 else 0.0
            row['sigma1_over_sigma2'] = s1 / s2 if s2 > 0 else np.inf
            rows.append(row)

df = pd.DataFrame(rows)
df['pct_rank'] = df['dprime_mean'].rank(pct=True)

# Percentile groups
df['group'] = 'middle'
df.loc[df['pct_rank'] >= 0.85, 'group'] = 'top_15'
df.loc[df['pct_rank'] <= 0.15, 'group'] = 'bottom_15'

print(f'Total triples: {len(df)}')
print(f'NaN d\' (any ISI): {df[[c for c in df.columns if "dprime_isi" in c]].isna().any(axis=1).sum()}')
df.head()

## 3. Heatmaps: d' as f(sigma1, sigma2) for each sigma0 slice

In [ ]:
n_isis = len(isi_values)
# Show a subset of sigma0 slices (up to 6)
s0_indices = np.linspace(0, len(sigma0_grid) - 1, min(6, len(sigma0_grid)), dtype=int)

for i_s0 in s0_indices:
    s0 = sigma0_grid[i_s0]
    fig, axes = plt.subplots(1, n_isis, figsize=(4 * n_isis, 3.5))
    if n_isis == 1:
        axes = [axes]
    fig.suptitle(f'sigma0 = {s0:.3f}', fontsize=14, fontweight='bold')

    for ax, isi in zip(axes, isi_values):
        isi = int(isi)
        mat = dprime[isi][i_s0]  # [n_s1, n_s2]
        im = ax.imshow(
            mat, origin='lower', aspect='auto',
            extent=[sigma2_grid[0], sigma2_grid[-1],
                    sigma1_grid[0], sigma1_grid[-1]],
            cmap='viridis',
        )
        ax.set_xlabel('sigma2')
        ax.set_ylabel('sigma1')
        ax.set_title(f'ISI={isi}')
        plt.colorbar(im, ax=ax, label="d'")

    plt.tight_layout()
    plt.show()

## 4. ISI decay curves by percentile group

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

isis_sorted = sorted([int(i) for i in isi_values])
colors = {'top_15': 'tab:green', 'middle': 'tab:blue', 'bottom_15': 'tab:red'}
labels = {'top_15': 'Top 15%', 'middle': 'Middle', 'bottom_15': 'Bottom 15%'}

for group_name in ['top_15', 'middle', 'bottom_15']:
    sub = df[df['group'] == group_name]
    means = [sub[f'dprime_isi{isi}'].mean() for isi in isis_sorted]
    sems = [sub[f'dprime_isi{isi}'].std() / np.sqrt(len(sub)) for isi in isis_sorted]
    ax.errorbar(isis_sorted, means, yerr=sems,
                marker='o', label=labels[group_name],
                color=colors[group_name], capsize=3)

ax.set_xlabel('ISI')
ax.set_ylabel("d'")
ax.set_title('ISI Decay by Performance Group')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Parameter sensitivity (marginal d' over each parameter)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, param, grid in zip(axes,
                            ['sigma0', 'sigma1', 'sigma2'],
                            [sigma0_grid, sigma1_grid, sigma2_grid]):
    for isi in isis_sorted:
        means = df.groupby(param)[f'dprime_isi{isi}'].mean()
        ax.plot(means.index, means.values, marker='o', label=f'ISI={isi}', markersize=4)
    ax.set_xlabel(param)
    ax.set_ylabel("mean d'")
    ax.set_title(f'Sensitivity to {param}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Parameter distributions by percentile group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, param in zip(axes, ['sigma0', 'sigma1', 'sigma2']):
    for group_name in ['top_15', 'middle', 'bottom_15']:
        sub = df[df['group'] == group_name]
        ax.hist(sub[param], bins=20, alpha=0.5,
                label=labels[group_name], color=colors[group_name])
    ax.set_xlabel(param)
    ax.set_ylabel('Count')
    ax.set_title(f'{param} distribution by group')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Score distributions (hit vs FA) from per-triple files

In [ ]:
per_triple_dir = os.path.join(SAVE_DIR, 'per_triple')

# Sample triples from each group
n_sample = 3
fig, axes = plt.subplots(3, n_sample, figsize=(4 * n_sample, 10))

for row_idx, group_name in enumerate(['top_15', 'middle', 'bottom_15']):
    sub = df[df['group'] == group_name].sample(n=min(n_sample, len(df[df['group'] == group_name])),
                                                random_state=42)
    for col_idx, (_, triple) in enumerate(sub.iterrows()):
        fname = f's0={triple["sigma0"]:.3f}_s1={triple["sigma1"]:.3f}_s2={triple["sigma2"]:.3f}.npz'
        fpath = os.path.join(per_triple_dir, fname)
        if not os.path.exists(fpath):
            axes[row_idx, col_idx].set_title('File not found')
            continue

        td = np.load(fpath, allow_pickle=True)
        ax = axes[row_idx, col_idx]

        # FA scores
        fa = td['fa_scores']
        ax.hist(fa, bins=30, alpha=0.5, label='FA', color='tab:red', density=True)

        # Hit scores (first available ISI)
        for isi in isi_values:
            key = f'hit_scores_isi{int(isi)}'
            if key in td and len(td[key]) > 0:
                ax.hist(td[key], bins=30, alpha=0.5,
                        label=f'Hit ISI={int(isi)}', density=True)

        ax.set_title(f'{labels[group_name]}\n'
                     f's0={triple["sigma0"]:.2f} s1={triple["sigma1"]:.2f} s2={triple["sigma2"]:.2f}',
                     fontsize=9)
        ax.legend(fontsize=6)
        ax.set_xlabel('Score (distance)')

plt.tight_layout()
plt.show()

## 8. ROC curves from per-triple files

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, group_name in zip(axes, ['top_15', 'middle', 'bottom_15']):
    sub = df[df['group'] == group_name].sample(n=min(3, len(df[df['group'] == group_name])),
                                                random_state=42)
    for _, triple in sub.iterrows():
        fname = f's0={triple["sigma0"]:.3f}_s1={triple["sigma1"]:.3f}_s2={triple["sigma2"]:.3f}.npz'
        fpath = os.path.join(per_triple_dir, fname)
        if not os.path.exists(fpath):
            continue

        td = np.load(fpath, allow_pickle=True)
        for isi in isi_values:
            isi = int(isi)
            fpr_key = f'roc_fpr_isi{isi}'
            tpr_key = f'roc_tpr_isi{isi}'
            if fpr_key in td and len(td[fpr_key]) > 0:
                auc_val = td.get(f'auc_isi{isi}', np.nan)
                label = f'ISI={isi} (AUC={float(auc_val):.3f})'
                ax.plot(td[fpr_key], td[tpr_key], label=label, alpha=0.7)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(labels[group_name])
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 9. sigma1 vs sigma2 regime analysis

How does the ratio sigma1/sigma2 relate to ISI decay steepness?

In [ ]:
# Filter out infinite ratios and zero sigmas
df_valid = df[(df['sigma1'] > 0) & (df['sigma2'] > 0)].copy()
df_valid['log_ratio'] = np.log10(df_valid['sigma1'] / df_valid['sigma2'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: log(sigma1/sigma2) vs delta_dp
ax = axes[0]
sc = ax.scatter(df_valid['log_ratio'], df_valid['delta_dp'],
                c=df_valid['dprime_mean'], cmap='viridis', s=8, alpha=0.5)
ax.set_xlabel('log10(sigma1 / sigma2)')
ax.set_ylabel("ISI decay (d'_min_ISI - d'_max_ISI)")
ax.set_title('Noise ratio vs ISI decay')
ax.axvline(0, color='k', ls='--', alpha=0.3)
plt.colorbar(sc, ax=ax, label="mean d'")
ax.grid(True, alpha=0.3)

# Scatter: sigma1 vs sigma2, colored by mean d'
ax = axes[1]
sc = ax.scatter(df_valid['sigma1'], df_valid['sigma2'],
                c=df_valid['dprime_mean'], cmap='viridis', s=8, alpha=0.5)
ax.plot([0, max(sigma1_grid[-1], sigma2_grid[-1])],
        [0, max(sigma1_grid[-1], sigma2_grid[-1])],
        'k--', alpha=0.3, label='sigma1 = sigma2')
ax.set_xlabel('sigma1')
ax.set_ylabel('sigma2')
ax.set_title('Parameter space colored by mean d\'')
ax.legend()
plt.colorbar(sc, ax=ax, label="mean d'")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Summary statistics

In [ ]:
print('Summary by group:')
print('=' * 70)
for group_name in ['top_15', 'middle', 'bottom_15']:
    sub = df[df['group'] == group_name]
    print(f'\n{labels[group_name]} ({len(sub)} triples):')
    print(f'  mean d\': {sub["dprime_mean"].mean():.4f} +/- {sub["dprime_mean"].std():.4f}')
    print(f'  median sigma0: {sub["sigma0"].median():.4f}')
    print(f'  median sigma1: {sub["sigma1"].median():.4f}')
    print(f'  median sigma2: {sub["sigma2"].median():.4f}')
    print(f'  mean ISI decay: {sub["delta_dp"].mean():.4f}')
    for isi in isis_sorted:
        print(f'    ISI={isi}: d\' = {sub[f"dprime_isi{isi}"].mean():.4f}')